In [2]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim 
from torchvision import datasets, transforms

In [4]:
# Normalize the train and testing data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [5]:
# Load MNIST Dataset
train_dataset = datasets.MNIST(root = './data', train = True, download = True, transform = transform)
test_dataset = datasets.MNIST(root = './data', train = False, download = True, transform = transform)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.29MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 343kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.75MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.17MB/s]


In [6]:
# Define data loaders
train_loader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size = 64, shuffle = True)
test_loader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size = 1000, shuffle = False)

In [7]:
# Neural Network Model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)    # Input layer to hidden layer
        self.fc2 = nn.Linear(128, 10)       # Hidden layer to output layer

    def forward(self, x):
        x = x.view(-1, 28*28)   # Flatten the input
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = Net()

In [8]:
# Loss Function & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum = 0.9)

In [10]:
# Model Training
for epoch in range(1, 6):   # 5 epochs
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()           # Zero the gradients
        output = model(data)            # Forward pass
        loss = criterion(output, target)# Compute Loss
        loss.backward()                 # Backward Pass
        optimizer.step()                # Update weights

        if batch_idx % 100 == 0:
            print(f'Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')

Epoch 1 [0/60000] Loss: 2.325429
Epoch 1 [6400/60000] Loss: 0.489534
Epoch 1 [12800/60000] Loss: 0.458025
Epoch 1 [19200/60000] Loss: 0.260916
Epoch 1 [25600/60000] Loss: 0.323603
Epoch 1 [32000/60000] Loss: 0.111270
Epoch 1 [38400/60000] Loss: 0.191852
Epoch 1 [44800/60000] Loss: 0.373985
Epoch 1 [51200/60000] Loss: 0.175903
Epoch 1 [57600/60000] Loss: 0.098050
Epoch 2 [0/60000] Loss: 0.159014
Epoch 2 [6400/60000] Loss: 0.097715
Epoch 2 [12800/60000] Loss: 0.203543
Epoch 2 [19200/60000] Loss: 0.302056
Epoch 2 [25600/60000] Loss: 0.147268
Epoch 2 [32000/60000] Loss: 0.274476
Epoch 2 [38400/60000] Loss: 0.139006
Epoch 2 [44800/60000] Loss: 0.054917
Epoch 2 [51200/60000] Loss: 0.102063
Epoch 2 [57600/60000] Loss: 0.123419
Epoch 3 [0/60000] Loss: 0.274115
Epoch 3 [6400/60000] Loss: 0.097434
Epoch 3 [12800/60000] Loss: 0.153287
Epoch 3 [19200/60000] Loss: 0.062501
Epoch 3 [25600/60000] Loss: 0.077515
Epoch 3 [32000/60000] Loss: 0.104075
Epoch 3 [38400/60000] Loss: 0.095629
Epoch 3 [44800/6

In [11]:
# Evaluate
model.eval()
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        pred = output.argmax(dim = 1, keepdim = True)   # Get the index of the max log-probability
        correct += pred.eq(target.view_as(pred)).sum().item()

print(f'Test Accuracy: {100. * correct / len(test_loader.dataset):.2f}%')

Test Accuracy: 97.85%


In [13]:
import tkinter as tk
from PIL import Image, ImageDraw, ImageOps
import torch
import cv2
import numpy as np

# Load your trained model
model.eval()

# Create a Tkinter window
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Draw a Digit")
        self.canvas = tk.Canvas(self, width=280, height=280, bg='white')
        self.canvas.pack()
        self.button = tk.Button(self, text="Predict", command=self.predict)
        self.button.pack()
        self.clear_button = tk.Button(self, text="Clear", command=self.clear)
        self.clear_button.pack()

        self.canvas.bind("<B1-Motion>", self.paint)
        self.image1 = Image.new("L", (280, 280), color=255)
        self.draw = ImageDraw.Draw(self.image1)

    def paint(self, event):
        x1, y1 = (event.x - 8), (event.y - 8)
        x2, y2 = (event.x + 8), (event.y + 8)
        self.canvas.create_oval(x1, y1, x2, y2, fill='black')
        self.draw.ellipse([x1, y1, x2, y2], fill=0)

    def clear(self):
        self.canvas.delete("all")
        self.draw.rectangle([0, 0, 280, 280], fill=255)

    def predict(self):
        # Resize and normalize the image to 28x28
        img = self.image1.resize((28, 28))
        img = ImageOps.invert(img)
        img = np.array(img).astype(np.float32) / 255.0
        img = torch.tensor(img).unsqueeze(0).unsqueeze(0)  # shape (1, 1, 28, 28)

        # Predict
        with torch.no_grad():
            output = model(img)
            pred = output.argmax(dim=1).item()
            print(f"Predicted digit: {pred}")
            self.title(f"Predicted: {pred}")

# Run the app
app = App()
app.mainloop()

Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 6
Predicted digit: 6
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 2
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 8
Predicted digit: 7
Predicted digit: 3
Predicted digit: 3
Predicted digit: 6
Predicted digit: 8
Predicted digit: 8
Predicted digit: 2
Predicted digit: 8
Predicted digit: 8
Predicted digit: 3
Predicted digit: 8
Predicted digit: 8
Predicted digit: 3
Predicted digit: 6
Predicted digit: 6
Predicted digit: 6
Predicted digit: 7
Predicted digit: 1
Predicted digit: 9
Predicted digit: 6
Predicted digit: 6
Predicted digit: 6
Predicted digit: 8
Predicted digit: 8
Predicted digit: 1
Predicted digit: 2
Predicted digit: 3
Predicted digit: 8
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 3
Predicted digit: 7
Predicted digit: 3
Predicted di